In [32]:
from langchain_groq import ChatGroq
from typing import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv


In [33]:
load_dotenv()

True

In [34]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str
    
    

In [35]:
model=ChatGroq(
    model='llama-3.3-70b-versatile'
)

In [36]:
def generate_joke(state:JokeState):
    prompt=f'Generate a short and easy to understandable joke on this {state['topic']}.'
    responce=model.invoke(prompt)
    
    return {'joke':responce}

In [37]:
def generate_explanation(state:JokeState):
    prompt=f'Generate a simple easy to understandable explanation of the given joke {state['joke']}.'
    responce=model.invoke(prompt)
    
    return {'explanation':responce}

In [38]:
thread_id = '1'

graph = StateGraph(JokeState)
check_point = InMemorySaver()

config = {
    'configurable': {
        'thread_id': thread_id
    }
}

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

workflow = graph.compile(checkpointer=check_point)

In [39]:
workflow.invoke({'topic':'cricket'},config=config)

{'topic': 'cricket',
 'joke': AIMessage(content='Why did the cricket go to the doctor?\n\nBecause it had a "bug" in its system and wanted to get a "pitch"-perfect checkup!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 47, 'total_tokens': 79, 'completion_time': 0.109372738, 'completion_tokens_details': None, 'prompt_time': 0.002342621, 'prompt_tokens_details': None, 'queue_time': 0.056930939, 'total_time': 0.111715359}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fc0dd-916b-7ed0-aeec-b5b79b875ef7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 32, 'total_tokens': 79}),
 'explanation': AIMessage(content='Let\'s break down this joke:\n\n**Joke:** Why did the cricket go to the doctor?\n**Answer:** Because it had a "bug" in its system and want